# W21 · ROS2 核心概念与工具链

> 阶段三（ROS2 Jazzy）第 1 讲。从这一讲开始，我们从「训练策略的算法工程师」
> 补全为「能把策略装进真实机器人系统的系统工程师」。

## 学习目标

学完本讲，你应该能够：

1. 说清楚 ROS1 → ROS2 的关键变化，以及 **DDS** 在其中的角色；
2. 区分 ROS2 的五大通信机制：**话题（Topic）、服务（Service）、动作（Action）、参数（Parameter）、Launch**，
   并为给定场景选出正确的机制；
3. 解释 **QoS** 的 `reliability / durability / history / depth` 四项设置及其组合效果；
4. 使用 `ros2` 命令行工具（`node / topic / service / action / param / run / launch`）观察与操控一个运行中的系统。

## ⚠️ 运行前提

本讲所有 `ros2` 命令都需要 **Ubuntu 24.04 + ROS2 Jazzy**（apt 安装，不能 pip），
安装步骤见 `docs/phase3_ros2.md`。本机没有 ROS2 环境，因此：

- `ros2` 命令统一写在 markdown 的 bash 代码块中，**复制到有 ROS2 的机器终端执行**；
- 纯 Python 概念演示（QoS 模拟等）使用项目环境真实执行。

## 1. 为什么机器人需要中间件？

一个真实机器人上有几十上百个并发进程：激光雷达驱动、IMU 驱动、定位、建图、规划、
控制、状态监控、日志……它们跑在不同的进程甚至不同的计算机上，却要毫秒级地交换数据。

**类比**：ROS2 之于机器人，就像「电话交换机 + 邮政系统 + 内部广播站」之于一家公司。
没有它，每两个部门之间都要拉一条专线（$N^2$ 条连接）；有了它，大家只需要知道
「广播频道名」（话题）或「分机号」（服务），交换机会自动完成寻址与传输。

ROS1（2010 年代）用自定义的 TCPROS 协议 + 一个中心化的 `roscore`（Master）做寻址。
这带来两个硬伤：

- **单点故障**：Master 挂掉，整个系统瘫痪；
- **无实时保障**：无法表达「这条消息必须送达」或「只要最新的就行」。

ROS2 把通信层整体换成了工业标准 **DDS（Data Distribution Service）**：
去中心化发现、可配置的 QoS、跨平台、有实时扩展。你不需要会配置 DDS，
但它决定了 ROS2 的一切行为，所以先看一个概念实验。

## 2. DDS 直觉：去中心化发现 + QoS 契约

**DDS 的两个核心思想**：

1. **去中心化发现（Discovery）**：没有 Master。每个节点启动后向局域网广播
   「我是谁、我发布/订阅什么」，其它节点听到后直接点对点建连。
2. **QoS 契约**：每一对发布-订阅之间协商一份「服务质量合同」，例如：
   - `reliability`：`reliable`（必须送达，像 TCP）还是 `best_effort`（尽力而为，像 UDP）；
   - `history` + `depth`：订阅者来不及时，发布端最多缓存最近几条（`keep_last`）；
   - `durability`：`volatile`（只管在线者）还是 `transient_local`（新订阅者能收到最后一条历史消息——地图这类「只发一次」的数据必须用它）。

下面的纯 Python 模拟帮你建立 `history/depth` 的直觉：**发布快、消费慢时，
队列深度决定了丢哪些消息**。代码不依赖 ROS2，可在本机直接运行。

In [1]:
"""纯 Python 模拟：QoS history=keep_last 的 depth 语义。

场景：IMU 以 100 Hz 发布，消费端处理一帧要 50 ms（≈20 Hz）。
队列相当于订阅端入口的缓冲区：满了以后，最老的消息被挤掉。
"""
from collections import deque


def simulate_keep_last(depth: int, n_pub: int = 20, consume_every: int = 5):
    """发布 n_pub 条消息，每发布 consume_every 条消费 1 条。"""
    queue = deque(maxlen=depth)  # maxlen 即「挤掉最老」的语义
    received, dropped = [], 0
    for seq in range(n_pub):
        if len(queue) == depth:
            dropped += 1  # 最老消息被新消息挤掉
        queue.append(seq)
        if (seq + 1) % consume_every == 0:
            received.append(queue.popleft())
    received.extend(queue)  # 结束后把缓冲区里剩余的都消费掉
    return received, dropped


for depth in (1, 5):
    received, dropped = simulate_keep_last(depth)
    print(f"depth={depth}: 实际消费 {received}")
    print(f"          被丢弃 {dropped} 条；消费端看到的是否总是最新值？{received[-1] == 19}")

depth=1: 实际消费 [4, 9, 14, 19]
          被丢弃 16 条；消费端看到的是否总是最新值？True
depth=5: 实际消费 [0, 5, 10, 15, 16, 17, 18, 19]
          被丢弃 12 条；消费端看到的是否总是最新值？True


**观察**：`depth=1` 时消费端每次都拿到「最新一帧」，代价是大量丢帧；
`depth=5` 时丢帧更少，但消费到的可能是「几帧之前的旧数据」。

工程经验法则：

| 数据类型 | reliability | history/depth | 理由 |
|----------|-------------|---------------|------|
| 传感器流（IMU/图像/点云） | `best_effort` | `keep_last`, depth 1~5 | 最新帧最有价值，丢帧可容忍；可靠传输的重传反而会引入延迟 |
| 指令/状态事件（急停、模式切换） | `reliable` | `keep_last`, depth 10 | 一条都不能丢 |
| 地图、静态描述（latched 数据） | `reliable` | `transient_local` | 后启动的节点也必须拿到 |

> ⚠️ QoS 是**协商契约**：发布端 `reliable` + 订阅端 `best_effort` 可以匹配（降级生效），
> 但发布端 `best_effort` + 订阅端 `reliable` **匹配失败，双方静默收不到消息**——
> 这是 ROS2 新手最常踩的坑之一。Gazebo 传感器默认发 `best_effort`，你的订阅端必须匹配。

## 3. 五大通信机制：什么时候用哪个

| 机制 | 模式 | 类比 | 典型用途 |
|------|------|------|----------|
| **Topic（话题）** | 单向流，多对多，异步 | 广播电台 | 传感器数据、速度指令、状态心跳 |
| **Service（服务）** | 请求-响应，一对一，快速 | 打前台电话问一句话 | 重置仿真、开关某个功能、查询配置 |
| **Action（动作）** | 目标-反馈-结果，可取消，长时间 | 外卖订单：下单→骑手位置实时更新→送达/取消 | 导航到某点、机械臂执行轨迹 |
| **Parameter（参数）** | 节点的键值配置，运行时可改 | 设备面板上的旋钮 | 控制器增益、阈值、话题名 |
| **Launch（启动文件）** | 一次性拉起多节点并注入参数 | 开工前的总电闸 | 整套系统的编排启动 |

选择启发式：**「连续数据流 → Topic；瞬时问答 → Service；有进度、可取消的长任务 → Action」**。

数学视角补充：动作 ≈ 把 RL 里的「一个 episode」暴露成网络接口——goal 是初始条件，
feedback 是中间状态 $s_t$，result 是终止回报。你在 W26 会发送 `NavigateToPose` 动作，
在 W27 会把 RL 策略包成订阅/发布话题的节点。

## 4. 动手实验：用 turtlesim 认识 CLI 工具链

> ⚠️ 以下命令在 **装好 ROS2 Jazzy 的机器**上执行（每条命令一个终端，先 `source /opt/ros/jazzy/setup.bash`）。
> 配套安装 `ros-jazzy-turtlesim` 与 `ros-jazzy-rqt-*`。

**Step 1 — 启动两只海龟与键盘控制：**

```bash
ros2 run turtlesim turtlesim_node        # 终端 1：仿真器（一个节点）
ros2 run turtlesim turtle_teleop_key     # 终端 2：键盘遥控（另一个节点）
```

**Step 2 — 观察系统（终端 3）：**

```bash
ros2 node list                 # 看到 /turtlesim 和 /teleop_turtle
ros2 node info /turtlesim      # 这个节点发布/订阅/服务了哪些接口
ros2 topic list -t             # 带消息类型列出话题
ros2 topic echo /turtle1/pose  # 实时打印位姿（x, y, theta, 线/角速度）
ros2 topic hz /turtle1/pose    # 统计发布频率（约 62 Hz）
```

**Step 3 — 主动操控：**

```bash
# 不用键盘，直接以 10 Hz 发布速度指令让乌龟画圆
ros2 topic pub --rate 10 /turtle1/cmd_vel geometry_msgs/msg/Twist \
  "{linear: {x: 2.0}, angular: {z: 1.8}}"

# 服务：再孵化一只乌龟
ros2 service call /spawn turtlesim/srv/Spawn "{x: 2.0, y: 2.0, theta: 0.0, name: 'turtle2'}"

# 动作：让乌龟转到绝对角度 90°（带反馈、可 Ctrl-C 取消）
ros2 action send_goal /turtle1/rotate_absolute turtlesim/action/RotateAbsolute \
  "{theta: 1.57}" --feedback

# 参数：在线修改背景色（蓝色通道拉满）
ros2 param set /turtlesim background_b 255
```

**预期输出要点**：

- `ros2 topic echo /turtle1/pose` 持续滚动 `x / y / theta`；画圆时 `theta` 匀速增长；
- `ros2 action send_goal ... --feedback` 持续打印 `remaining: x.xx`（剩余弧度）直到为 0；
- `ros2 param set` 返回 `Set parameter successful`，背景立即变紫。

做完这套实验，你就已经用过了本讲的全部五种机制（launch 在下一讲登场）。

## 5. 本讲小结

- ROS2 = **节点（Node）** 组成的计算图，节点间用 **Topic/Service/Action** 通信，
  用 **Parameter** 配置，用 **Launch** 编排；
- 底层是 **DDS**：去中心化发现 + QoS 契约；QoS 不匹配是「静默故障」头号来源；
- CLI 工具链（`ros2 node/topic/service/action/param`）是排查一切系统问题的第一手段。

## ✏️ 练习

> 练习 1–3 需要 ROS2 环境（可集中在机动周完成）；练习 4 是纯 Python，本机可做。

### 练习 1（★，约 15 分钟）：接口普查

启动 turtlesim 后，用 CLI 完成一份「接口普查表」，交付为 markdown 表格：
`/turtlesim` 节点的全部 Publisher / Subscription / Service / Action Server，
并给每个接口标注它属于五大机制中的哪一种、消息类型是什么。

### 练习 2（★★，约 20 分钟）：不用键盘画一个正方形

只用 `ros2 topic pub`（或 `ros2 action send_goal`）组合，让 turtle1 走出一个
边长约 2 的正方形轨迹。交付：你实际执行的命令序列 + `ros2 topic echo /turtle1/pose`
中关键时刻的位姿截图/文本，并解释为什么开环控制（盲发速度）很难画准——
这正是 RL 闭环策略存在的意义。

### 练习 3（★★，约 25 分钟）：为场景选机制

为下列场景各选一种通信机制并给出 QoS 建议，交付决策表：
(a) 100 Hz 的 IMU 数据；(b) 「把机械臂回零位」按钮；(c) 「导航到 (3, 5)」任务；
(d) 控制器 PID 增益；(e) 一张只发布一次的静态地图；(f) 急停信号。

### 练习 4（★★★，约 40 分钟，纯 Python）：扩展 QoS 模拟器

在本讲 `simulate_keep_last` 的基础上实现 `simulate_qos(reliability, depth, loss_p)`：
`best_effort` 时每条消息以概率 `loss_p` 在网络中随机丢失，`reliable` 时重传保证送达
但带来 1 个时间步的额外延迟。分别在 `loss_p ∈ {0, 0.1, 0.3}` 下统计：
消费端收到的消息数、收到的最大「数据年龄」（当前序号 - 收到消息的序号）。
交付：代码 + 结果表格 + 一段话结论（什么时候 best_effort 反而更优）。

## 参考答案

<details>
<summary>参考答案</summary>

**练习 1**：`ros2 node info /turtlesim` 输出中应包含——
Publishers：`/turtle1/pose (turtlesim/msg/Pose)`、`/parameter_events`、`/rosout`；
Subscriptions：`/turtle1/cmd_vel (geometry_msgs/msg/Twist)`；
Services：`/clear`、`/kill`、`/reset`、`/spawn`、`/turtle1/set_pen`、`/turtle1/teleport_absolute`、`/turtle1/teleport_relative`；
Action Servers：`/turtle1/rotate_absolute (turtlesim/action/RotateAbsolute)`。
其中 pose/cmd_vel 是话题（数据流），kill/spawn/teleport 等瞬时操作是服务，
rotate_absolute 有进度反馈、可取消，是动作。

**练习 2**：示例命令序列（每段后用 `Ctrl-C` 停止）：

```bash
ros2 topic pub --once /turtle1/cmd_vel geometry_msgs/msg/Twist "{linear: {x: 2.0}}"   # 前进
ros2 action send_goal /turtle1/rotate_absolute turtlesim/action/RotateAbsolute "{theta: 1.5708}"
# 重复 4 次
```

开环画不准的原因：没有反馈校正，轮胎打滑、数值积分误差、发布频率抖动都会累积——
这正是闭环控制/RL 策略（观测→动作→再观测）要解决的问题。

**练习 3**：(a) Topic + `best_effort`/depth 1；(b) Service（瞬时、要确认结果）；
(c) Action（长任务、要反馈和取消）；(d) Parameter；(e) Topic + `transient_local` durability；
(f) Topic + `reliable`（且通常走独立的高优先级通道， worst case 应设计成硬件层兜底）。

**练习 4**（参考实现）：

```python
import random
from collections import deque

def simulate_qos(reliability, depth, loss_p, n_pub=200, consume_every=5, seed=0):
    rng = random.Random(seed)
    queue = deque(maxlen=depth)
    in_flight = []          # (到达时刻, seq)，reliable 的重传延迟 1 步
    received, ages = [], []
    for t in range(n_pub):
        queue.append(t)     # 新消息进入发布端缓存（挤掉最老）
        # 网络传输：尝试把队首（最老未发）送出去
        if queue:
            seq = queue[0]
            if reliability == "best_effort":
                if rng.random() >= loss_p:
                    in_flight.append((t, seq))
                queue.popleft()
            else:  # reliable：丢则重传，下一步必达
                if rng.random() >= loss_p:
                    in_flight.append((t, seq))
                    queue.popleft()
                # 否则留在队列里，下一步重传
        # 消费端每 consume_every 步消费一次
        if (t + 1) % consume_every == 0 and in_flight:
            t_arr, seq = in_flight.pop(0)
            received.append(seq)
            ages.append(t - t_arr)
    return len(received), (max(ages) if ages else 0)
```

典型结论：`loss_p` 越大，`best_effort` 收到的新鲜数据越多（年龄小），
而 `reliable` 保证不丢但数据越来越旧——高频传感流宁要「新而稀疏」，不要「全而陈旧」。
</details>

## 延伸阅读

- [ROS2 Jazzy 官方文档](https://docs.ros.org/en/jazzy/)（本阶段所有内容的第一信源）
- [官方教程：Introducing turtlesim & rqt](https://docs.ros.org/en/jazzy/Tutorials/Beginner-CLI-Tools/Introducing-Turtlesim/Introducing-Turtlesim.html)
- [ROS2 Actions 设计文档](https://design.ros2.org/articles/actions.html)（理解动作机制的设计动机）
- [eProsima Fast DDS 文档](https://fast-dds.docs.eprosima.com/)（Jazzy 默认 DDS 实现，QoS 细节可查这里）
- [鱼香 ROS《ROS2 机器人开发：从入门到实践》](https://fishros.com/d2lros2/)（中文系统教程，适合交叉阅读）
- 下一讲预告：W22 用 `rclpy` 亲手写出发布/订阅节点与 Launch 文件。